# Daily Dhan → BigQuery (thin notebook)

**Git holds only the reusable functions** (the `dhan-pipeline` package).
**This notebook holds every variable** — dates, Google Sheet, project, tables, creds.
Edit the VARIABLES cell, then run.

In [ ]:
# 1. Install the shared functions from GitHub (pin a tag/commit in production)
!pip install -q "git+https://github.com/rajatjain1992/dhan-pipeline.git"

In [ ]:
# 2. Mount Drive (for the service-account JSON)
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# 3. ===== VARIABLES — edit these, this is the only place values live =====
from dhan_pipeline import Config, recent_window

# --- dates: either use the last-N-days helper, or hardcode strings ---
FROM_DATE, TO_DATE = recent_window(days=4)      # -> >= 2 trading days
# FROM_DATE, TO_DATE = '2026-07-27', '2026-07-28'  # or set explicitly

cfg = Config(
    # Dhan API creds (kept in Colab secrets, not in the notebook text)
    dhan_client_id     = userdata.get('DHAN_CLIENT_ID'),
    dhan_access_token  = userdata.get('DHAN_ACCESS_TOKEN'),
    service_account_file = '/content/drive/MyDrive/Colab Notebooks/rajat-trade-c411eaec7c51.json',

    # BigQuery target
    project_id       = 'rajat-trade',
    dataset_id       = 'stock_data_set',
    daily_table      = 'stock_daily_prices_dhan',
    staging_table    = 'stock_daily_prices_dhan_staging',
    flag_table       = 'corporate_action_flags',
    instrument_table = 'instrument_list',

    # Google Sheet with the scrip list
    sheet_key          = '1aoEgOhQkAAv8b2NqAWtZUYXG41rOal77i0XasevyNtE',
    list_worksheet     = 'my_list',
    negative_worksheet = 'Negative List',
)
print(FROM_DATE, '->', TO_DATE)

In [ ]:
# 4. Run: fetch window -> split-check 2nd-last day -> upsert last day
from dhan_pipeline import run_daily

result = run_daily(cfg, FROM_DATE, TO_DATE)
result['flags']   # scrips flagged for a suspected split/adjustment

In [ ]:
# 5. (Optional) subset the scrip list, e.g. only intraday=yes_1
from dhan_pipeline import load_scrip_mapping, gspread_client, subset
gc = gspread_client(cfg)
mapping = load_scrip_mapping(cfg, gc)
intraday = subset(mapping, 'intraday', ['yes_1'])
# result = run_daily(cfg, FROM_DATE, TO_DATE, scrip_mapping=intraday)